# Convertible bonds

**Prerequisites:** **06**. **`convertible_bond`** combines **rates + credit + equity vol**. In this build, `credit_curve_id` must reference a **discount** (risky curve), not a hazard curve.


## Concept

- **Conversion ratio / price:** shares per bond vs effective conversion strike.
- **Conversion premium:** equity upside vs straight bond floor.
- **Bond floor:** straight debt component before conversion optionality.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, MarketContext

print("Imports OK (convertibles).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

cb = json.dumps(
    {
        "type": "convertible_bond",
        "spec": {
            "id": "CB-TECH-5Y",
            "notional": {"amount": "1000000", "currency": "USD"},
            "issue_date": "2024-01-15",
            "maturity": "2029-01-15",
            "discount_curve_id": "USD-IG",
            "credit_curve_id": "USD-CREDIT-BBB",
            "conversion": {
                "ratio": 25.0,
                "policy": "Voluntary",
                "anti_dilution": "None",
                "dividend_adjustment": "None",
            },
            "underlying_equity_id": "TECH",
            "fixed_coupon": {
                "coupon_type": "Cash",
                "rate": 0.02,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "end_of_month": False,
                "payment_lag_days": 0,
                "stub": "None",
            },
            "attributes": {"tags": [], "meta": {}},
            "pricing_overrides": {},
        },
    }
)

try:
    validate_instrument_json(cb)
    print("convertible JSON OK")
except Exception as exc:
    print("validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85)],
    day_count="act_365f",
)
ig = DiscountCurve(
    "USD-IG",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.965), (5.0, 0.80)],
    day_count="act_365f",
)
risky = DiscountCurve(
    "USD-CREDIT-BBB",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.94), (5.0, 0.72)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois).insert(ig).insert(risky)
state = json.loads(mc.to_json())
state["surfaces"] = [
    {
        "id": "TECH-VOL",
        "expiries": [0.25, 0.5, 1.0, 2.0],
        "strikes": [20.0, 40.0, 60.0],
        "secondary_axis": "strike",
        "interpolation_mode": "vol",
        "vols_row_major": [0.35] * (4 * 3),
    }
]
state.setdefault("prices", {})
state["prices"]["TECH"] = {"price": {"amount": 40.0, "currency": "USD"}}
market_json = json.dumps(state)
print("surfaces:", len(state["surfaces"]))


## Pricing


In [ ]:
try:
    out = price_instrument_with_metrics(cb, market_json, AS_OF_STR, model="discounting")
    print(ValuationResult.from_json(out))
except Exception as exc:
    print("price:", type(exc).__name__, exc)


## Metrics


In [ ]:
try:
    out = price_instrument_with_metrics(
        cb,
        market_json,
        AS_OF_STR,
        model="discounting",
        metrics=["bond_floor", "conversion_premium", "conversion_value", "delta"],
    )
    vr = ValuationResult.from_json(out)
    for m in ("bond_floor", "conversion_premium", "conversion_value", "delta"):
        v = vr.get_metric(m)
        if v is not None:
            print(m, ":", v)
except Exception as exc:
    print("metrics:", type(exc).__name__, exc)


## Takeaways

- **Risky discount curve** as `credit_curve_id` matches the current loader expectations.
- **Equity spot + vol surface** IDs must exist under `prices` / `surfaces`.
- Conversion metrics decompose **fixed-income floor** vs **equity optionality**.
